# Three-Model Featurewise MLP Quality Scoring

This notebook is the earlier three-model Featurewise MLP quality-scoring workflow. It uses separate neural score heads for geometry, appearance, and densification. The model scores candidate settings and then selects the candidate with the highest predicted relative quality score.


## 0. Install Notebook Dependencies

Run this setup cell before the training cells. `tqdm` is used for live Ridge lambda-search and MLP epoch progress bars.


In [ ]:
%pip install tqdm


## 1. Configuration

Set paths and high-level options. The defaults point to the current workflow artifacts in this repository plus the registered test pipeline folder.

In [ ]:
from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

WORKSPACE_ROOT = Path(r"D:\bimba3d-re")
TRAINING_DATA_ID = "training_data_20260710_183008_final_offline_data_june_27-training-data"
TRAINING_DATA_ROWS_PATH = (
    WORKSPACE_ROOT
    / "bimba3d_backend"
    / "data"
    / "workflow"
    / "training_data"
    / TRAINING_DATA_ID
    / "rows.json"
)
MODELS_INDEX_PATH = WORKSPACE_ROOT / "bimba3d_backend" / "data" / "workflow" / "models" / "models_index.json"

# Change this if your test pipeline is stored somewhere else.
TEST_PIPELINE_FOLDER = Path(r"E:\Thesis\PipelineTests\Final_test_June_27")

# Stored fallback for prediction: if no explicit test candidate grid is provided,
# the model checks this many multiplier values inside each group.
CANDIDATE_POINTS = 30
RIDGE_LAMBDA_CANDIDATES = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
RUN_LABEL = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = WORKSPACE_ROOT / "notebooks" / "outputs" / f"ridge_replica_{RUN_LABEL}"

print("Configuration")
print("-------------")
print(f"Workspace root      : {WORKSPACE_ROOT}")
print(f"Training rows path  : {TRAINING_DATA_ROWS_PATH}  exists={TRAINING_DATA_ROWS_PATH.exists()}")
print(f"Models index path   : {MODELS_INDEX_PATH}  exists={MODELS_INDEX_PATH.exists()}")
print(f"Test pipeline folder: {TEST_PIPELINE_FOLDER}  exists={TEST_PIPELINE_FOLDER.exists()}")
print(f"Output directory    : {OUT_DIR}")
print(f"Candidate points    : {CANDIDATE_POINTS}")

## 2. MLP Hyperparameters

These are the Featurewise MLP defaults used in this notebook. MLP training uses PyTorch random initialization and split behavior, so exact numeric equality requires the same seed/checkpoint.

In [ ]:
# MLP hyperparameters mirror bimba3d_backend/worker/ai_input_modes/featurewise_mlp_runtime.py.
MLP_HIDDEN = 8
MLP_DROPOUT = 0.2
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-3
MLP_EPOCHS = 1000
MLP_PATIENCE = 50
OUT_DIR = WORKSPACE_ROOT / "notebooks" / "outputs" / f"mlp_replica_{RUN_LABEL}"

print("MLP hyperparameters")
print("-------------------")
print(f"hidden={MLP_HIDDEN}, dropout={MLP_DROPOUT}, lr={MLP_LR}, weight_decay={MLP_WEIGHT_DECAY}")
print(f"epochs={MLP_EPOCHS}, patience={MLP_PATIENCE}")
print(f"Output directory: {OUT_DIR}")

## 3. Schema Constants

The MLP uses the same three parameter groups and descriptor subsets as Ridge, but learns a nonlinear score function over candidate actions.

In [ ]:
# These constants are kept here so the notebook is self-contained:
# - bimba3d_backend/scripts/train_offline_models.py
# - bimba3d_backend/worker/ai_input_modes/featurewise_*_runtime.py
#
# The model learns three group-level multipliers and expands them to the
# individual gsplat parameters used by the training run.
FEATUREWISE_GROUP_FEATURES = {
    "geometry_lr_mult": [
        "intercept",
        "focal_norm",
        "gsd_norm",
        "overlap_proxy",
        "coverage_spread",
        "camera_angle_bucket",
        "heading_consistency",
        "blur_motion_risk",
        "terrain_roughness_proxy",
        "vegetation_cover",
    ],
    "appearance_lr_mult": [
        "intercept",
        "iso_norm",
        "image_resolution_norm",
        "blur_motion_risk",
        "texture_density",
        "vegetation_cover",
        "vegetation_complexity",
    ],
    "densification_mult": [
        "intercept",
        "gsd_norm",
        "overlap_proxy",
        "coverage_spread",
        "camera_angle_bucket",
        "texture_density",
        "blur_motion_risk",
        "terrain_roughness_proxy",
        "vegetation_complexity",
    ],
}

GROUP_KEYS = ["geometry_lr_mult", "appearance_lr_mult", "densification_mult"]

GROUP_BOUNDS = {
    "geometry_lr_mult": (0.5, 2.0),
    "appearance_lr_mult": (0.5, 2.0),
    "densification_mult": (0.7, 1.4285714286),
}

GROUP_BOUND_ALIASES = {
    "geometry_lr": "geometry_lr_mult",
    "geometry": "geometry_lr_mult",
    "appearance_lr": "appearance_lr_mult",
    "appearance": "appearance_lr_mult",
    "scale_lr": "densification_mult",
    "densification": "densification_mult",
    "densification_lr": "densification_mult",
}

GROUP_MULTIPLIERS_MAP = {
    "geometry_lr_mult": ["position_lr_init_mult", "scaling_lr_mult", "rotation_lr_mult"],
    "appearance_lr_mult": ["feature_lr_mult", "opacity_lr_mult", "lambda_dssim_mult"],
    "densification_mult": ["densify_grad_threshold_mult", "opacity_threshold_mult"],
}

GSPLAT_PARAMETER_DEFAULTS = {
    "feature_lr": 0.0025,
    "position_lr_init": 0.00016,
    "position_lr_final": 1.6e-06,
    "scaling_lr": 0.005,
    "opacity_lr": 0.05,
    "rotation_lr": 0.001,
    "densify_grad_threshold": 0.0002,
    "opacity_threshold": 0.005,
    "lambda_dssim": 0.2,
}

LOG_TRANSFORM_FEATURES = {"focal_norm", "gsd_norm", "iso_norm"}
SCALER_STD_FLOOR = 0.01

print("Group schema")
print("------------")
for group in GROUP_KEYS:
    print(f"{group:22s} features={len(FEATUREWISE_GROUP_FEATURES[group]):2d} members={GROUP_MULTIPLIERS_MAP[group]}")

## 4. Shared Helpers

These helpers load JSON artifacts, normalize bounds, read Training Data rows, read test-project descriptors, and apply final multipliers to gsplat defaults.

In [ ]:
def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def normalise_group_bounds(bounds: dict[str, Any] | None = None) -> dict[str, tuple[float, float]]:
    """Return positive multiplier bounds keyed by group names."""
    out = dict(GROUP_BOUNDS)
    if not isinstance(bounds, dict):
        return out
    for raw_key, raw_value in bounds.items():
        group_key = GROUP_BOUND_ALIASES.get(str(raw_key), str(raw_key))
        if group_key not in out or not isinstance(raw_value, (list, tuple)) or len(raw_value) < 2:
            continue
        try:
            lo, hi = float(raw_value[0]), float(raw_value[1])
        except (TypeError, ValueError):
            continue
        if not math.isfinite(lo) or not math.isfinite(hi) or lo <= 0.0 or hi <= 0.0:
            continue
        if hi < lo:
            lo, hi = hi, lo
        out[group_key] = (lo, hi)
    return out


def clamp_float(value: float, lo: float, hi: float) -> float:
    return float(min(max(float(value), float(lo)), float(hi)))


def _extract_raw_feature(features: dict[str, Any], name: str) -> float:
    """Extract the same raw feature proxy used by model training."""
    if name == "focal_norm":
        return float(features.get("focal_length_mm", 24.0) or 24.0)
    if name == "iso_norm":
        return float(features.get("iso", 400.0) or 400.0)
    if name == "image_resolution_norm":
        w = float(features.get("img_width_median", 4000.0) or 4000.0)
        h = float(features.get("img_height_median", 3000.0) or 3000.0)
        return (w * h) / 1e6
    if name == "gsd_norm":
        return float(features.get("gsd_median", 0.05) or 0.05)
    if name == "overlap_proxy":
        return float(features.get("overlap_proxy", 0.5) or 0.5)
    if name == "coverage_spread":
        return float(features.get("coverage_spread", 0.5) or 0.5)
    if name == "camera_angle_bucket":
        return float(features.get("camera_angle_bucket", 0) or 0)
    if name == "heading_consistency":
        return float(features.get("heading_consistency", 0.5) or 0.5)
    if name == "texture_density":
        return float(features.get("texture_density", 0.5) or 0.5)
    if name == "blur_motion_risk":
        return float(features.get("blur_motion_risk", 0.5) or 0.5)
    if name == "terrain_roughness_proxy":
        return float(features.get("terrain_roughness_proxy", features.get("terrain_roughness", 0.5)) or 0.5)
    if name == "vegetation_cover":
        value = features.get("vegetation_cover_percentage", features.get("vegetation_cover", 0.5))
        return float(value or 0.5)
    if name == "vegetation_complexity":
        value = features.get("vegetation_complexity_score", features.get("vegetation_complexity", 0.5))
        return float(value or 0.5)
    return 0.0


def _apply_log_transform(name: str, value: float) -> float:
    return math.log(max(value, 1e-9)) if name in LOG_TRANSFORM_FEATURES else value


def load_training_rows(path: Path) -> list[dict[str, Any]]:
    """Match model_training._filter_rows: skip baseline rows and require score/multipliers."""
    payload = load_json(path)
    rows = payload.get("rows", payload) if isinstance(payload, dict) else payload
    out: list[dict[str, Any]] = []
    for row in rows if isinstance(rows, list) else []:
        if not isinstance(row, dict):
            continue
        if row.get("is_baseline_row") or row.get("is_baseline_run"):
            continue
        if not isinstance(row.get("x_features"), dict) or not isinstance(row.get("selected_multipliers"), dict):
            continue
        if not isinstance(row.get("relative_quality_score"), (int, float)):
            continue
        out.append(row)
    return out


def load_project_features(folder: Path) -> dict[str, dict[str, Any]]:
    """Load test-pipeline project EXIF descriptors from project-level exif_features.json files."""
    projects: dict[str, dict[str, Any]] = {}
    if not folder.exists():
        return projects
    for project_dir in sorted(folder.iterdir()):
        if not project_dir.is_dir() or project_dir.name.startswith("_"):
            continue
        exif_path = project_dir / "exif_features.json"
        if not exif_path.exists():
            continue
        payload = load_json(exif_path)
        features = payload.get("features") if isinstance(payload, dict) else None
        if isinstance(features, dict) and features:
            projects[project_dir.name] = {
                "project_name": project_dir.name,
                "project_dir": project_dir,
                "features": features,
            }
    return projects


def source_bounds_from_model_index() -> dict[str, tuple[float, float]]:
    """Use the source log-multiplier bounds stored with the prepared data when available."""
    index = load_json(MODELS_INDEX_PATH) if MODELS_INDEX_PATH.exists() else []
    existing = next(
        (
            item
            for item in index
            if item.get("model_family") == "featurewise_ridge_regression"
            and item.get("source_training_data_id") == TRAINING_DATA_ID
        ),
        None,
    )
    raw_bounds = (existing or {}).get("config", {}).get("log_multiplier_bounds")
    return normalise_group_bounds(raw_bounds)


def final_effective_params(selected_multipliers: dict[str, float]) -> dict[str, float]:
    """Apply selected group multipliers to gsplat defaults exactly as the preset resolver does."""
    params = dict(GSPLAT_PARAMETER_DEFAULTS)
    mapping = {
        "position_lr_init": "position_lr_init_mult",
        "position_lr_final": "position_lr_init_mult",
        "scaling_lr": "scaling_lr_mult",
        "rotation_lr": "rotation_lr_mult",
        "feature_lr": "feature_lr_mult",
        "opacity_lr": "opacity_lr_mult",
        "lambda_dssim": "lambda_dssim_mult",
        "densify_grad_threshold": "densify_grad_threshold_mult",
        "opacity_threshold": "opacity_threshold_mult",
    }
    for param_key, mult_key in mapping.items():
        params[param_key] = float(params[param_key]) * float(selected_multipliers.get(mult_key, 1.0))
    return params

## 5. Load Data

This block loads reusable Training Data rows and test project EXIF descriptors. It prints a compact summary before training.

In [ ]:
training_rows = load_training_rows(TRAINING_DATA_ROWS_PATH)
test_projects = load_project_features(TEST_PIPELINE_FOLDER)
bounds = source_bounds_from_model_index()

print("Loaded data")
print("-----------")
print(f"Training rows       : {len(training_rows)}")
print(f"Training projects   : {len({row.get('project_name') for row in training_rows})}")
print(f"Test projects       : {len(test_projects)}")
print(f"Multiplier bounds   : {bounds}")

if training_rows:
    scores = [float(row["relative_quality_score"]) for row in training_rows]
    print(f"Score mean/std/min/max: {np.mean(scores):.6f} / {np.std(scores):.6f} / {np.min(scores):.6f} / {np.max(scores):.6f}")

if test_projects:
    print("\nTest projects with descriptors:")
    for name, info in test_projects.items():
        print(f"  {name:40s} feature_count={len(info['features'])}")
else:
    print("\nNo test project descriptors found. Update TEST_PIPELINE_FOLDER before running prediction cells.")

## 6. Dataset Tables

Reporting block only. It prints and displays dataset counts for auditability.

In [ ]:
project_counts = {}
for row in training_rows:
    project_counts[row.get("project_name", "unknown")] = project_counts.get(row.get("project_name", "unknown"), 0) + 1

dataset_summary = {
    "training_rows": len(training_rows),
    "training_projects": len(project_counts),
    "test_projects": len(test_projects),
    "candidate_points": CANDIDATE_POINTS,
}
print(json.dumps(dataset_summary, indent=2))

if pd is not None and project_counts:
    df_project_counts = (
        pd.DataFrame([{"project": key, "rows": value} for key, value in project_counts.items()])
        .sort_values("rows", ascending=False)
        .reset_index(drop=True)
    )
    display(df_project_counts.head(20))

## 7. MLP Training and Prediction Functions

Working code for the three-head Featurewise MLP, score-design tensors, training loop, checkpoint writing, and candidate-grid selection.

In [ ]:
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
except Exception as exc:
    HAS_TORCH = False
    print("PyTorch unavailable:", exc)



if HAS_TORCH:
    class FeaturewiseGroupMLP(nn.Module):
        """Small per-group score MLP used by this notebook."""

        def __init__(self, input_dim: int, hidden: int = 8, dropout: float = 0.2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, hidden // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, x):
            return self.net(x)


    class FeaturewiseMLP(nn.Module):
        """Three independent score heads: geometry, appearance, densification."""

        def __init__(self, geo_dim: int, app_dim: int, den_dim: int, hidden: int = 8, dropout: float = 0.2):
            super().__init__()
            self.geometry_net = FeaturewiseGroupMLP(geo_dim, hidden, dropout)
            self.appearance_net = FeaturewiseGroupMLP(app_dim, hidden, dropout)
            self.densification_net = FeaturewiseGroupMLP(den_dim, hidden, dropout)

        def forward(self, x_geo, x_app, x_den):
            return torch.cat(
                [self.geometry_net(x_geo), self.appearance_net(x_app), self.densification_net(x_den)],
                dim=1,
            )


def build_score_design_vector(x_context: np.ndarray, action_log: float) -> np.ndarray:
    interactions = x_context[1:] * float(action_log)
    return np.concatenate([x_context, np.array([action_log, action_log * action_log]), interactions])


def build_featurewise_vector(features: dict[str, Any], group_key: str) -> np.ndarray:
    values = []
    for name in FEATUREWISE_GROUP_FEATURES[group_key]:
        values.append(1.0 if name == "intercept" else _apply_log_transform(name, _extract_raw_feature(features, name)))
    return np.array(values, dtype=np.float32)


def build_featurewise_score_tensor(features: dict[str, Any], group_key: str, action_log: float) -> np.ndarray:
    return build_score_design_vector(build_featurewise_vector(features, group_key), float(action_log)).astype(np.float32)


def extract_group_action_log(row: dict[str, Any], group_key: str, bounds: dict[str, tuple[float, float]]) -> float | None:
    """Prefer selected_log_multipliers, then derive log from selected multipliers."""
    selected_logs = row.get("selected_log_multipliers") if isinstance(row.get("selected_log_multipliers"), dict) else {}
    for key in [group_key, *GROUP_MULTIPLIERS_MAP[group_key]]:
        value = selected_logs.get(key)
        if isinstance(value, (int, float)) and math.isfinite(float(value)):
            lo, hi = bounds[group_key]
            return clamp_float(float(value), math.log(lo), math.log(hi))

    selected = row.get("selected_multipliers")
    if not isinstance(selected, dict):
        return None
    direct = selected.get(group_key)
    if isinstance(direct, (int, float)) and math.isfinite(float(direct)) and float(direct) > 0:
        lo, hi = bounds[group_key]
        return math.log(clamp_float(float(direct), lo, hi))
    values = [
        float(selected[key])
        for key in GROUP_MULTIPLIERS_MAP[group_key]
        if isinstance(selected.get(key), (int, float)) and float(selected[key]) > 0
    ]
    if not values:
        return None
    lo, hi = bounds[group_key]
    return math.log(clamp_float(float(np.mean(values)), lo, hi))


def train_featurewise_mlp(rows: list[dict[str, Any]], save_dir: Path, group_bounds=None) -> dict[str, Any]:
    """Train the featurewise score MLP using the notebook tensors and training loop."""
    if not HAS_TORCH:
        return {"trained": False, "error": "PyTorch unavailable"}
    bounds = normalise_group_bounds(group_bounds)
    x_geo, x_app, x_den, targets = [], [], [], []
    for row in rows:
        features = row.get("x_features")
        score = row.get("relative_quality_score")
        if not isinstance(features, dict) or not isinstance(score, (int, float)):
            continue
        logs = {group: extract_group_action_log(row, group, bounds) for group in GROUP_KEYS}
        if any(value is None for value in logs.values()):
            continue
        x_geo.append(build_featurewise_score_tensor(features, "geometry_lr_mult", logs["geometry_lr_mult"]))
        x_app.append(build_featurewise_score_tensor(features, "appearance_lr_mult", logs["appearance_lr_mult"]))
        x_den.append(build_featurewise_score_tensor(features, "densification_mult", logs["densification_mult"]))
        targets.append(np.array([float(score), float(score), float(score)], dtype=np.float32))
    if not targets:
        return {"trained": False, "error": "No valid rows"}

    x_geo_t = torch.tensor(np.array(x_geo), dtype=torch.float32)
    x_app_t = torch.tensor(np.array(x_app), dtype=torch.float32)
    x_den_t = torch.tensor(np.array(x_den), dtype=torch.float32)
    y_t = torch.tensor(np.array(targets), dtype=torch.float32)
    n = len(y_t)

    # Torch default RNG controls split and initialization after the seed is set.
    perm = torch.randperm(n)
    split = max(1, int(0.8 * n))
    train_idx, val_idx = perm[:split], perm[split:]

    model = FeaturewiseMLP(x_geo_t.shape[1], x_app_t.shape[1], x_den_t.shape[1], MLP_HIDDEN, MLP_DROPOUT)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    best_val_loss, best_state, best_epoch, patience_counter = float("inf"), None, 0, 0
    train_losses, val_losses = [], []
    print("Featurewise MLP training status")
    print("-------------------------------")
    print(f"valid rows={n}, train rows={len(train_idx)}, validation rows={len(val_idx)}")
    print(f"dims: geo={x_geo_t.shape[1]}, appearance={x_app_t.shape[1]}, densification={x_den_t.shape[1]}")
    print(f"optimizer: Adam lr={MLP_LR}, weight_decay={MLP_WEIGHT_DECAY}, max_epochs={MLP_EPOCHS}, patience={MLP_PATIENCE}")
    epoch_progress = tqdm(range(MLP_EPOCHS), desc="Featurewise MLP epochs", unit="epoch")
    for epoch in epoch_progress:
        model.train()
        optimizer.zero_grad()
        loss = ((model(x_geo_t[train_idx], x_app_t[train_idx], x_den_t[train_idx]) - y_t[train_idx]) ** 2).mean()
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            val_loss = (
                ((model(x_geo_t[val_idx], x_app_t[val_idx], x_den_t[val_idx]) - y_t[val_idx]) ** 2).mean().item()
                if len(val_idx)
                else float(loss.item())
            )
        val_losses.append(float(val_loss))
        if hasattr(epoch_progress, "set_postfix"):
            epoch_progress.set_postfix(
                train_loss=f"{float(loss.item()):.6f}",
                val_loss=f"{float(val_loss):.6f}",
                best=f"{float(best_val_loss):.6f}" if math.isfinite(best_val_loss) else "inf",
                patience=patience_counter,
            )
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = float(val_loss)
            best_state = {key: value.clone() for key, value in model.state_dict().items()}
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= MLP_PATIENCE:
                print(f"Early stopping at epoch {epoch + 1}; best_epoch={best_epoch}, best_val_loss={best_val_loss:.6f}")
                break

    print(f"Finished {len(train_losses)} epochs; best_epoch={best_epoch}; best_val_loss={best_val_loss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    model_dir = save_dir / "featurewise_mlp"
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "featurewise.pt"
    torch.save(
        {
            "state_dict": model.state_dict(),
            "model_type": "featurewise_mlp",
            "geo_dim": int(x_geo_t.shape[1]),
            "app_dim": int(x_app_t.shape[1]),
            "den_dim": int(x_den_t.shape[1]),
            "hidden": MLP_HIDDEN,
            "dropout": MLP_DROPOUT,
            "learning_rate": MLP_LR,
            "weight_decay": MLP_WEIGHT_DECAY,
            "max_epochs": MLP_EPOCHS,
            "early_stopping_patience": MLP_PATIENCE,
            "candidate_points": 30,
            "training_samples": int(n),
            "log_multiplier_bounds": {key: [float(bounds[key][0]), float(bounds[key][1])] for key in GROUP_KEYS},
        },
        model_path,
    )
    metadata = {
        "model_type": "featurewise_mlp",
        "score_key": "relative_quality_score",
        "geo_dim": int(x_geo_t.shape[1]),
        "app_dim": int(x_app_t.shape[1]),
        "den_dim": int(x_den_t.shape[1]),
        "hidden": MLP_HIDDEN,
        "dropout": MLP_DROPOUT,
        "training_samples": int(n),
        "epochs_trained": len(train_losses),
        "max_epochs": MLP_EPOCHS,
        "best_epoch": best_epoch,
        "early_stopping_patience": MLP_PATIENCE,
        "best_val_loss": best_val_loss,
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "learning_rate": MLP_LR,
        "weight_decay": MLP_WEIGHT_DECAY,
        "candidate_points": 30,
        "log_multiplier_bounds": {key: [float(bounds[key][0]), float(bounds[key][1])] for key in GROUP_KEYS},
        "train_losses": train_losses,
        "val_losses": val_losses,
    }
    metadata_path = model_dir / "featurewise_metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return {"trained": True, "model_path": str(model_path), "metadata_path": str(metadata_path), **metadata}


def select_mlp_multipliers(checkpoint_path: Path, features: dict[str, Any], candidate_logs_by_group=None) -> dict[str, Any]:
    """Scan candidate actions and select the max predicted score for each group."""
    if not HAS_TORCH:
        raise RuntimeError("PyTorch unavailable")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = FeaturewiseMLP(
        int(checkpoint["geo_dim"]),
        int(checkpoint["app_dim"]),
        int(checkpoint["den_dim"]),
        int(checkpoint.get("hidden", 8)),
        float(checkpoint.get("dropout", 0.2)),
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    bounds = normalise_group_bounds(checkpoint.get("log_multiplier_bounds"))
    candidate_points = int(checkpoint.get("candidate_points", 30))
    group_multipliers, group_logs, score_spreads, candidate_score_checks, has_signal = {}, {}, {}, {}, {}

    with torch.no_grad():
        for group in GROUP_KEYS:
            lo, hi = bounds[group]
            raw_logs = (candidate_logs_by_group or {}).get(group) if isinstance(candidate_logs_by_group, dict) else None
            logs = (
                np.array([clamp_float(float(value), math.log(lo), math.log(hi)) for value in raw_logs if isinstance(value, (int, float))], dtype=np.float32)
                if isinstance(raw_logs, list) and raw_logs
                else np.linspace(math.log(lo), math.log(hi), max(5, candidate_points), dtype=np.float32)
            )
            batch = torch.tensor(np.stack([build_featurewise_score_tensor(features, group, float(action_log)) for action_log in logs]), dtype=torch.float32)
            z_geo = torch.zeros((len(logs), int(checkpoint["geo_dim"])), dtype=torch.float32)
            z_app = torch.zeros((len(logs), int(checkpoint["app_dim"])), dtype=torch.float32)
            z_den = torch.zeros((len(logs), int(checkpoint["den_dim"])), dtype=torch.float32)
            if group == "geometry_lr_mult":
                scores = model(batch, z_app, z_den)[:, 0].cpu().numpy()
            elif group == "appearance_lr_mult":
                scores = model(z_geo, batch, z_den)[:, 1].cpu().numpy()
            else:
                scores = model(z_geo, z_app, batch)[:, 2].cpu().numpy()
            spread = float(np.max(scores) - np.min(scores)) if len(scores) else 0.0
            score_spreads[group] = spread
            selected_log = 0.0 if spread < 1e-6 else float(logs[int(np.argmax(scores))])
            selected_mult = clamp_float(math.exp(selected_log), lo, hi)
            selected_log = math.log(selected_mult)
            group_multipliers[group] = selected_mult
            group_logs[group] = selected_log
            has_signal[group] = spread >= 1e-6
            candidate_score_checks[group] = [
                {
                    "candidate_log_multiplier": float(logs[index]),
                    "candidate_multiplier": float(math.exp(float(logs[index]))),
                    "predicted_score": float(scores[index]),
                    "selected": bool(abs(float(logs[index]) - selected_log) < 1e-12),
                }
                for index in range(len(logs))
            ]

    selected_multipliers, selected_logs = {}, {}
    for group, members in GROUP_MULTIPLIERS_MAP.items():
        for member in members:
            selected_multipliers[member] = group_multipliers[group]
            selected_logs[member] = group_logs[group]
    return {
        "selected_preset": "featurewise_mlp",
        "selected_multipliers": selected_multipliers,
        "selected_log_multipliers": selected_logs,
        "group_multipliers": group_multipliers,
        "score_spreads": score_spreads,
        "candidate_score_checks": candidate_score_checks,
        "candidate_points": candidate_points,
        "has_signal": all(has_signal.values()),
        "n_runs": int(checkpoint.get("training_samples") or 0),
    }


## 8. Train Featurewise MLP

Working block: train the MLP score model and write the checkpoint/metadata artifacts.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

mlp_result = train_featurewise_mlp(training_rows, OUT_DIR, bounds)
print("MLP training result")
print("-------------------")
print(json.dumps({key: value for key, value in mlp_result.items() if key not in {"train_losses", "val_losses"}}, indent=2))

if not mlp_result.get("trained"):
    raise RuntimeError(mlp_result.get("error", "MLP training failed"))

mlp_model_path = Path(mlp_result["model_path"])
mlp_metadata_path = Path(mlp_result["metadata_path"])
train_losses = mlp_result["train_losses"]
val_losses = mlp_result["val_losses"]

## 9. Training Loss Chart

Visualization block only. It plots train and validation loss from the MLP training run.

In [ ]:
if plt is None:
    print("matplotlib is not available.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(train_losses, label="train loss")
    ax.plot(val_losses, label="validation loss")
    ax.set_title("Featurewise MLP Training Curve")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 10. Predict Test-Project Multipliers

Working block: scan candidate actions with the trained MLP and store selected multipliers, descriptors, candidate-score checks, and final values.

In [ ]:
prediction_rows = []
for project_name, info in test_projects.items():
    features = info["features"]
    pred = select_mlp_multipliers(mlp_model_path, features)
    row = {
        "project_name": project_name,
        "model_id": "notebook_mlp_replica",
        "model_family": "featurewise_mlp",
        "mode": "exif_compact_featurewise",
        "status": "ok",
        "features": features,
        **pred,
    }
    row["effective_params"] = final_effective_params(row["selected_multipliers"])
    prediction_rows.append(row)

preview_path = OUT_DIR / "prediction_preview.json"
preview_path.write_text(json.dumps({"rows": prediction_rows}, indent=2), encoding="utf-8")

print("Predictions complete")
print("--------------------")
print(f"Prediction rows: {len(prediction_rows)}")
print("Preview artifact:", preview_path)

## 11. Prediction Table

Reporting block: show one row per project with selected group multipliers, log multipliers, and final parameter values.

In [ ]:
summary_rows = []
for row in prediction_rows:
    multipliers = row["selected_multipliers"]
    logs = row["selected_log_multipliers"]
    effective = row["effective_params"]
    summary_rows.append(
        {
            "project": row["project_name"],
            "geo_mult": multipliers["position_lr_init_mult"],
            "app_mult": multipliers["feature_lr_mult"],
            "dens_mult": multipliers["densify_grad_threshold_mult"],
            "geo_log": logs["position_lr_init_mult"],
            "app_log": logs["feature_lr_mult"],
            "dens_log": logs["densify_grad_threshold_mult"],
            "position_lr_init": effective["position_lr_init"],
            "feature_lr": effective["feature_lr"],
            "densify_grad_threshold": effective["densify_grad_threshold"],
            "candidate_points": row["candidate_points"],
            "has_signal": row["has_signal"],
        }
    )

if pd is not None:
    df_predictions = pd.DataFrame(summary_rows)
    display(df_predictions)
else:
    print(json.dumps(summary_rows, indent=2))

## 12. Prediction Chart

Visualization block only. It compares selected group multipliers across test projects.

In [ ]:
if plt is None:
    print("matplotlib is not available.")
elif not prediction_rows:
    print("No prediction rows to visualize.")
else:
    labels = [row["project_name"] for row in prediction_rows]
    x = np.arange(len(labels))
    width = 0.25
    geo = [row["selected_multipliers"]["position_lr_init_mult"] for row in prediction_rows]
    app = [row["selected_multipliers"]["feature_lr_mult"] for row in prediction_rows]
    dens = [row["selected_multipliers"]["densify_grad_threshold_mult"] for row in prediction_rows]

    fig, ax = plt.subplots(figsize=(max(9, len(labels) * 0.7), 4.5))
    ax.bar(x - width, geo, width, label="Geometry")
    ax.bar(x, app, width, label="Appearance")
    ax.bar(x + width, dens, width, label="Densification")
    ax.axhline(1.0, color="black", linewidth=1, linestyle="--", alpha=0.5)
    ax.set_title("Predicted Group Multipliers by Test Project")
    ax.set_ylabel("Multiplier")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 13. Candidate Score Table

Reporting block: inspect the full candidate score list for one project/group. Change `PROJECT_NAME` and `GROUP` to inspect another curve.

In [ ]:
PROJECT_NAME = prediction_rows[0]["project_name"] if prediction_rows else None
GROUP = "geometry_lr_mult"

if not prediction_rows:
    print("No predictions available.")
else:
    selected_row = next((row for row in prediction_rows if row["project_name"] == PROJECT_NAME), prediction_rows[0])
    checks = selected_row["candidate_score_checks"].get(GROUP, [])
    candidate_table = [
        {
            "candidate": index + 1,
            "log_multiplier": item["candidate_log_multiplier"],
            "multiplier": item["candidate_multiplier"],
            "predicted_score": item["predicted_score"],
            "selected": item["selected"],
        }
        for index, item in enumerate(checks)
    ]
    print(f"Candidate inspection: project={selected_row['project_name']} group={GROUP}")
    if pd is not None:
        df_candidates = pd.DataFrame(candidate_table)
        display(df_candidates)
    else:
        print(json.dumps(candidate_table, indent=2))

## 14. Candidate Score Curve

Visualization block only. It plots the candidate score surface and marks the selected multiplier.

In [ ]:
if plt is None:
    print("matplotlib is not available.")
elif not prediction_rows:
    print("No predictions available.")
else:
    xs = [row["multiplier"] for row in candidate_table]
    ys = [row["predicted_score"] for row in candidate_table]
    selected = [row for row in candidate_table if row["selected"]]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(xs, ys, marker="o")
    if selected:
        ax.scatter([selected[0]["multiplier"]], [selected[0]["predicted_score"]], s=90, color="#f59e0b", label="selected", zorder=5)
    ax.set_xscale("log")
    ax.set_title(f"Candidate Score Curve: {PROJECT_NAME} / {GROUP}")
    ax.set_xlabel("Multiplier")
    ax.set_ylabel("Predicted score")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 15. Artifact Summary

Final reporting block showing the files written by the notebook.

In [ ]:
artifact_summary = {
    "output_dir": str(OUT_DIR),
    "prediction_preview": str(preview_path),
}
if "ridge_path" in globals():
    artifact_summary["ridge_model"] = str(ridge_path)
if "mlp_model_path" in globals():
    artifact_summary["mlp_model"] = str(mlp_model_path)
if "mlp_metadata_path" in globals():
    artifact_summary["mlp_metadata"] = str(mlp_metadata_path)

print(json.dumps(artifact_summary, indent=2))